# Example 4.5 / 1: Declaring Sparse Links

Companion notebook for **4.5 Sets of Pairs** in the AMPL lab Documentation, section "Declaring Sparse Links". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Declaring Sparse Links

The data supplies only usable origin–destination links. Assigning a Python list of tuples gives
the pair set; a dictionary with tuple keys gives the parameter indexed over it.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ORIGINS;
set DESTINATIONS;
set LINKS within {ORIGINS, DESTINATIONS};
param cost {LINKS} >= 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ORIGINS"] = ["GARY", "CLEV"]
ampl.set["DESTINATIONS"] = ["DET", "LAN", "STL"]
ampl.set["LINKS"] = [("GARY", "DET"), ("GARY", "STL"), ("CLEV", "LAN")]
ampl.param["cost"] = {("GARY", "DET"): 14, ("GARY", "STL"): 16, ("CLEV", "LAN"): 12}

In [5]:
# run.py
print(ampl.set["LINKS"].to_list())
print(ampl.param["cost"].to_dict())

[('GARY', 'DET'), ('GARY', 'STL'), ('CLEV', 'LAN')]
{('CLEV', 'LAN'): 12, ('GARY', 'DET'): 14, ('GARY', 'STL'): 16}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ORIGINS := GARY CLEV;
set DESTINATIONS := DET LAN STL;
set LINKS := (GARY,DET) (GARY,STL) (CLEV,LAN);
param cost := GARY DET 14  GARY STL 16  CLEV LAN 12;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display LINKS, cost;

set LINKS := (GARY,DET) (GARY,STL) (CLEV,LAN);

cost :=
CLEV LAN   12
GARY DET   14
GARY STL   16
;

